### Improving grpo for reinforcement learning--->  rlvr

#### downloading previous chapter notebook

In [1]:
from pathlib import Path
import requests


def download_from_github(rel_path, out=None):

    github_raw_base = (
        "https://raw.githubusercontent.com/"
        "shaheennabi/open-posttraining-system/main/"
    )

    rel_path = Path(rel_path)

    out = Path(out) if out is not None else Path(rel_path.name)

    if out.exists():
        print(f"{out} already exists")
        return out

    r = requests.get(
        github_raw_base + rel_path.as_posix()
    )
    r.raise_for_status()

    out.write_bytes(r.content)

    print(f"Downloaded {out}")

    return out

In [2]:
download_from_github(
    "training_reasoning_models_with_reinforcement_learning/rlvr_grpo_training_with_no_kl.py"
)

rlvr_grpo_training_with_no_kl.py already exists


PosixPath('rlvr_grpo_training_with_no_kl.py')

In [ ]:
### rlvr_grpo training run
!uv run rlvr_grpo_training_with_no_kl.py --steps 500 --max_new_tokens 1024

In [3]:
## log file (download from previous chapter)
download_from_github(
    "training_reasoning_models_with_reinforcement_learning/train_rlvr_grpo_metrics.csv"
)

train_rlvr_grpo_metrics.csv already exists


PosixPath('train_rlvr_grpo_metrics.csv')

In [4]:
import pandas as pd

df = pd.read_csv(
    "train_rlvr_grpo_metrics.csv",
    header=None,
    names=[
        "step",
        "eval_freq",
        "loss",
        "reward_avg",
        "avg_response_len",
    ],
)

print(df.head())

   step  eval_freq      loss  reward_avg  avg_response_len
0     1         50 -1.239144        0.25            287.75
1     2         50 -3.203884        0.25            295.00
2     3         50 -0.000000        1.00            152.25
3     4         50 -0.000000        1.00            129.00
4     5         50 -5.945770        0.75            143.50


#### Inspecting the grpo training run (from sebastian raschka's notebook)

In [5]:
download_from_github(
    "evaluating_reasoning_models/eval_math_500.py"
)

eval_math_500.py already exists


PosixPath('eval_math_500.py')

In [ ]:

from pathlib import Path

# Start a fresh checkpoint-evaluation table for this experiment run.
Path("eval_metrics.csv").unlink(missing_ok=True)

# Evaluate every local checkpoint produced by the GRPO run.
# This writes one eval_metrics.csv row per checkpoint step (50, 100, 150, ...).
!uv run eval_math_500.py --dataset_size 50 --checkpoint_glob "checkpoints/**/*.safetensors" --metrics_csv "eval_metrics.csv"



### Merge training metrics with checkpoint evaluations

The training CSV is dense: it has one row per optimization step. The MATH-500 evaluation CSV is sparse: it has one row per evaluated checkpoint. A single row in `eval_metrics.csv` means only one checkpoint was evaluated, not that the full experiment has only one training step. The merge below keeps the dense training trace and attaches evaluation accuracy only at checkpoint steps.


In [ ]:

from pathlib import Path
import pandas as pd


def read_training_metrics(path="train_rlvr_grpo_metrics.csv"):
    path = Path(path)
    first_line = path.read_text(encoding="utf-8").splitlines()[0]

    if first_line.startswith("step,"):
        df = pd.read_csv(path)
    else:
        raw = pd.read_csv(path, header=None)
        if raw.shape[1] == 5:
            raw.columns = [
                "step",
                "total_steps",
                "loss",
                "reward_avg",
                "avg_response_len",
            ]
        elif raw.shape[1] == 7:
            raw.columns = [
                "step",
                "total_steps",
                "loss",
                "reward_avg",
                "tokens_per_sec",
                "avg_response_len",
                "eval_acc",
            ]
        else:
            raise ValueError(f"Unexpected training metrics width: {raw.shape[1]}")
        df = raw

    df["step"] = pd.to_numeric(df["step"], errors="coerce").astype("Int64")
    for col in ["loss", "reward_avg", "tokens_per_sec", "avg_response_len", "eval_acc"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["step"]).copy()


def read_eval_metrics(path="eval_metrics.csv"):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(
            columns=[
                "step",
                "math500_eval_acc",
                "num_correct",
                "num_examples",
                "eval_avg_response_len",
                "runtime_minutes",
                "jsonl_path",
            ]
        )

    df = pd.read_csv(path)
    df["step"] = pd.to_numeric(df["step"], errors="coerce").astype("Int64")
    rename_map = {
        "eval_acc": "math500_eval_acc",
        "avg_response_len": "eval_avg_response_len",
    }
    df = df.rename(columns=rename_map)
    for col in ["math500_eval_acc", "num_correct", "num_examples", "eval_avg_response_len", "runtime_minutes"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    keep_cols = [
        col for col in [
            "step",
            "math500_eval_acc",
            "num_correct",
            "num_examples",
            "eval_avg_response_len",
            "runtime_minutes",
            "jsonl_path",
        ]
        if col in df.columns
    ]
    return (
        df[keep_cols]
        .dropna(subset=["step"])
        .sort_values("step")
        .drop_duplicates(subset=["step"], keep="last")
    )


train_df = read_training_metrics("train_rlvr_grpo_metrics.csv")
eval_df = read_eval_metrics("eval_metrics.csv")

merged_df = train_df.merge(eval_df, on="step", how="left")
merged_df.to_csv("merged_grpo_metrics.csv", index=False)

print(f"training rows: {len(train_df)}")
print(f"eval rows: {len(eval_df)}")
print("Saved: merged_grpo_metrics.csv")
display(merged_df.tail())


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd


def moving_average(series, window_fraction=0.20):
    window = max(1, int(len(series) * window_fraction))
    return series.rolling(window=window, min_periods=1).mean()


metrics = pd.read_csv("merged_grpo_metrics.csv")
metrics["step"] = pd.to_numeric(metrics["step"], errors="coerce")
metrics = metrics.dropna(subset=["step"]).sort_values("step")

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
axes = axes.ravel()

plot_specs = [
    ("loss", "GRPO loss", "line"),
    ("reward_avg", "Average rollout reward", "line"),
    ("avg_response_len", "Training response length", "line"),
    ("math500_eval_acc", "MATH-500 checkpoint accuracy", "eval"),
]

for ax, (column, title, kind) in zip(axes, plot_specs):
    if column not in metrics.columns or metrics[column].dropna().empty:
        ax.set_title(f"{title} (no data)")
        ax.axis("off")
        continue

    values = pd.to_numeric(metrics[column], errors="coerce")
    mask = values.notna()

    if kind == "eval":
        ax.plot(metrics.loc[mask, "step"], values[mask], marker="o", linewidth=1.5)
        ax.set_ylim(0, 1)
    else:
        ax.plot(metrics["step"], values, alpha=0.28, linewidth=1)
        ax.plot(metrics["step"], moving_average(values), linewidth=2)

    ax.set_title(title)
    ax.set_xlabel("Training step")
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


### Experiment readout

The GRPO run should be read as two linked logs. The dense training log tracks optimization behavior every step: loss, rollout reward, and generated response length. The evaluation log is intentionally sparse because MATH-500 is run only at saved checkpoints. When `eval_metrics.csv` has one row, that means only one checkpoint, such as step 50, has been evaluated. To produce a real evaluation curve, run the checkpoint-glob eval cell after multiple checkpoints exist.

In the merged table, `math500_eval_acc` is therefore expected to be blank for most training steps and populated only at checkpoint steps. The plot treats training metrics as continuous traces and checkpoint accuracy as measured points. This avoids inventing evaluation data between checkpoints while still showing how the policy-training signals relate to downstream MATH-500 accuracy.


### Tracking more advanced Grpo performance metrics

* Advantage tracking

In [ ]:
import torch

def compute_advantage_stats(rewards_list):
    # This is what we already compute in GRPO:
    rewards = torch.tensor(rewards_list)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # These are the new statistics we add:
    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

* Entropy tracking